# Making SIMSOPT GPU native: end-to-end augmented Lagrangian

Select **Runtime > Change runtime type > GPU**, then run all cells. This qualification starts both solvers from the same circular order-8 engineering coils. It compares the original CPU Python/SciPy augmented-Lagrangian against a single compiled GPU-native outer/inner augmented-Lagrangian executable. There is deliberately no refinement phase. The CPU reference can take approximately 15 minutes on Colab; keep the tab connected until the archive downloads.

In [ ]:
import shutil
import subprocess

nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi is None:
    raise RuntimeError("No NVIDIA GPU is attached. Select Runtime > Change runtime type > T4 GPU, disconnect the old runtime, and reconnect.")
subprocess.run([nvidia_smi], check=True)

In [ ]:
import importlib
import os
import sys
from pathlib import Path

repo = Path("/content/simsopt")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "gpu-native-objective", "https://github.com/PedroFranciscoGil/simsopt.git", str(repo)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "switch", "gpu-native-objective"], cwd=repo, check=True)
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "pytest"], cwd=repo, check=True)
os.chdir(repo)
source_root = repo / "src"
sys.path.insert(0, str(source_root))
for module_name in tuple(sys.modules):
    if module_name == "simsopt" or module_name.startswith("simsopt."):
        del sys.modules[module_name]
importlib.invalidate_caches()
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print(revision)

In [ ]:
import jax
import simsopt
from simsopt.gpu import backend_report

resolved_package = Path(simsopt.__file__).resolve()
assert source_root in resolved_package.parents, resolved_package
report = backend_report()
print(report)
assert jax.default_backend() == "gpu", report
assert jax.config.jax_enable_x64, report
assert any(device.platform == "gpu" for device in jax.devices()), report

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/gpu/test_device_augmented_lagrangian.py", "tests/gpu/test_augmented_lagrangian.py", "tests/gpu/test_end_to_end_device_augmented_lagrangian.py"], cwd=repo, check=True)

In [ ]:
artifact_root = Path("/content/simsopt-end-to-end-device-al")
artifact_root.mkdir(exist_ok=True)
result_path = artifact_root / "end-to-end-device-augmented-lagrangian.json"
env = os.environ.copy()
env["OMP_NUM_THREADS"] = "1"
env["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
subprocess.run([sys.executable, "benchmarks/gpu/benchmark_end_to_end_device_augmented_lagrangian.py", "--problem", "engineering", "--max-outer-iterations", "8", "--max-inner-iterations", "300", "--mu-init", "10", "--mu-max", "1e12", "--tau", "2", "--gradient-tolerance", "1e-8", "--constraint-tolerance", "1e-6", "--inner-stationarity-relative-tolerance", "0.01", "--inner-acceptance-mode", "budgeted", "--history-size", "20", "--cpu-maxcor", "100", "--max-line-search-iterations", "50", "--constraint-transform-epsilon", "0.1", "--constraint-scale-reduction-factor", "0.5", "--target-relative-tolerance", "0.10", "--quadratic-flux-target", "1e-5", "--current-scale", "100000", "--minimum-current-ratio", "0.5", "--maximum-current-ratio", "1.5", "--curve-coefficient-bound-radius", "0.25", "--target-tile-size", "1024", "--source-tile-size", "4320", "--gpu-warm-repeats", "3", "--output", str(result_path)], cwd=repo, env=env, check=True)

In [ ]:
import json

result = json.loads(result_path.read_text())
assert result["schema_version"] == 3
assert result["workflow"] == "end_to_end_device_augmented_lagrangian"
assert result["method"]["refinement_performed"] is False
assert result["cpu"]["execution_platform"] == "cpu"
assert result["gpu_native"]["execution_platform"] == "gpu"
assert result["gpu_native"]["device_resident"]
assert result["gpu_native"]["host_callbacks"] == 0
assert result["method"]["gpu_outer_loop_device_resident"]
assert result["method"]["matched_design_envelope_bounds"]
assert result["method"]["inner_stationarity_is_diagnostic"]
assert result["method"]["target_aware_outer_checkpoint_selection"]
assert result["design_envelope"]["curve_coefficient_bound_radius_m"] == 0.25
for backend in ("cpu", "gpu_native"):
    metrics = result[backend]["final_metrics"]
    assert {"objective", "quadratic_flux", "normalized_normal_field", "coil_constraints"} <= metrics.keys()
    optimization = result[backend]["optimization"]
    assert len(optimization["history"]) == optimization["outer_iterations"]
    assert all(item["optimizer_variables"] for item in optimization["history"])
    selection = result[backend]["checkpoint_selection"]
    assert selection["candidate_count"] == len(selection["candidates"])
for final_design in result["visualizations"].values():
    for artifact in final_design.values():
        if not isinstance(artifact, str) or not artifact.endswith((".vts", ".vtu")):
            continue
        path = Path(artifact)
        if not path.is_absolute():
            path = artifact_root / path
        assert path.is_file() and path.stat().st_size > 0, path
print(json.dumps({"scientifically_validated": result["scientifically_validated"], "technical_trajectory_agreement": result["technical_trajectory_agreement"], "comparison": result["comparison"], "cpu_final_metrics": result["cpu"]["final_metrics"], "gpu_native_final_metrics": result["gpu_native"]["final_metrics"]}, indent=2))

In [ ]:
from google.colab import files

archive = shutil.make_archive("/content/simsopt-end-to-end-device-al", "zip", artifact_root)
files.download(archive)

## What to send back

Send the downloaded `simsopt-end-to-end-device-al.zip`. The archive is retained whether or not the no-refinement endpoints satisfy every physical target.